In [4]:
import numpy as np
import trimesh
import trimesh.creation as tc
import json
import csv
import os

def leer_csv(ruta):
    coords = []
    with open(ruta, newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            coords.append([float(row['x']), float(row['y']), float(row['z'])])
    return coords

#Leer desde JSON (Bonus)
def leer_json(ruta):
    with open(ruta) as f:
        data = json.load(f)
    return [[d['x'], d['y'], d['z']] for d in data]


def crear_objeto(tipo, posicion, escala=1.0, color=None):
    """
    Crea una primitiva 3D con trimesh.
    tipo: 'cubo' | 'esfera' | 'cilindro'
    posicion: [x, y, z]
    escala: float
    color: [R, G, B, A] con valores 0-255, o None para color por defecto
    """
    if tipo == 'cubo':
        mesh = tc.box(extents=[escala, escala, escala])
        color_def = [255, 100, 100, 255]   # rojo
    elif tipo == 'esfera':
        mesh = tc.icosphere(radius=escala * 0.5, subdivisions=3)
        color_def = [100, 160, 255, 255]   # azul
    elif tipo == 'cilindro':
        mesh = tc.cylinder(radius=escala * 0.3, height=escala * 1.2, sections=32)
        color_def = [100, 220, 140, 255]   # verde
    else:
        raise ValueError(f'Tipo desconocido: {tipo}')

    # Aplicar color
    mesh.visual.face_colors = color if color else color_def

    # Aplicar traslacion
    matriz = np.eye(4)
    matriz[:3, 3] = posicion
    mesh.apply_transform(matriz)

    return mesh



def generar_escena(coordenadas, escala_base=1.0, modo_tipo='mix', modo_color='por_tipo'):
    """
    Genera una lista de meshes 3D a partir de una lista de coordenadas.
    modo_tipo: 'mix' | 'cubo' | 'esfera' | 'cilindro'
    modo_color: 'por_tipo' | 'por_altura' | 'aleatorio'
    """
    tipos_ciclo = ['esfera', 'cubo', 'cilindro']
    meshes = []
    registro = []

    for i, coord in enumerate(coordenadas):
        x, y, z = coord

        # Decidir tipo (condicional / bucle)
        if modo_tipo == 'mix':
            tipo = tipos_ciclo[i % 3]
        else:
            tipo = modo_tipo

        # Variar escala segun posicion
        escala = escala_base * (0.7 + abs(y) * 0.1 + np.random.uniform(0, 0.4))

        # Decidir color
        if modo_color == 'por_tipo':
            col = None   # usa color por defecto del tipo
        elif modo_color == 'por_altura':
            t = min(max((y + 5) / 10, 0), 1)
            col = [int(t*255), 80, int((1-t)*255), 255]
        else:  # aleatorio
            col = [np.random.randint(80,255) for _ in range(3)] + [255]

        posicion = [x, y + escala * 0.5, z]
        m = crear_objeto(tipo, posicion, escala, col)
        meshes.append(m)
        registro.append({'id': i, 'tipo': tipo, 'x': round(x,2),
                          'y': round(y,2), 'z': round(z,2), 'escala': round(escala,2)})

    return meshes, registro



def exportar_escena(meshes, registro, carpeta='exportados', fmt='obj'):
    """
    Exporta todos los meshes de la escena.
    fmt: 'obj' | 'stl' | 'glb'
    """
    os.makedirs(carpeta, exist_ok=True)

    # Combinar todos los meshes en una escena unica
    escena = trimesh.Scene()
    for i, m in enumerate(meshes):
        escena.add_geometry(m, node_name=f'obj_{i:03d}')

    # Exportar segun formato
    if fmt == 'obj':
        ruta = os.path.join(carpeta, 'escena.obj')
        escena.export(ruta)
    elif fmt == 'stl':
        # STL exporta mesh combinado
        combinado = trimesh.util.concatenate(meshes)
        ruta = os.path.join(carpeta, 'escena.stl')
        combinado.export(ruta)
    elif fmt == 'glb':
        ruta = os.path.join(carpeta, 'escena.glb')
        escena.export(ruta)
    else:
        raise ValueError(f'Formato no soportado: {fmt}')

    print(f'Escena exportada en: {ruta}')

    # Guardar registro JSON (Bonus)
    ruta_json = os.path.join(carpeta, 'registro.json')
    with open(ruta_json, 'w') as f:
        json.dump(registro, f, indent=2)
    print(f'Registro JSON guardado en: {ruta_json}')



if __name__ == '__main__':
    # Elegir fuente de datos
    coordenadas = leer_csv('datos.csv')   # cambiar por leer_json('datos.jsoj')

    # Generar escena
    meshes, registro = generar_escena(
        coordenadas,
        escala_base=1.0,
        modo_tipo='mix',        # 'mix' | 'cubo' | 'esfera' | 'cilindro'
        modo_color='por_tipo'   # 'por_tipo' | 'por_altura' | 'aleatorio'
    )

    print(f'Objetos generados: {len(meshes)}')
    for r in registro[:5]:
        print(r)

    # Exportar
    exportar_escena(meshes, registro, carpeta='exportados', fmt='obj')
    exportar_escena(meshes, registro, carpeta='exportados', fmt='glb')


Objetos generados: 8
{'id': 0, 'tipo': 'esfera', 'x': 0.0, 'y': 0.0, 'z': 0.0, 'escala': 0.82}
{'id': 1, 'tipo': 'cubo', 'x': 2.5, 'y': 0.0, 'z': -1.0, 'escala': 0.74}
{'id': 2, 'tipo': 'cilindro', 'x': -3.0, 'y': 0.5, 'z': 2.0, 'escala': 1.02}
{'id': 3, 'tipo': 'esfera', 'x': 1.0, 'y': 1.0, 'z': 3.0, 'escala': 0.98}
{'id': 4, 'tipo': 'cubo', 'x': -1.5, 'y': 0.0, 'z': -2.5, 'escala': 0.75}
Escena exportada en: exportados\escena.obj
Registro JSON guardado en: exportados\registro.json
Escena exportada en: exportados\escena.glb
Registro JSON guardado en: exportados\registro.json
